# DS5 Statistical Mechanics (StM) — Stage 4 Step 1

**Notebook:** `41_ds5_statistical_mechanics.ipynb`  
**Model family:** Statistical Mechanics (StM)  
**Dataset:** DS5 — German Adolescents RCT (N=573)  
**Stage:** 4 · Step 1 — Theoretical and Conceptual Modeling  
**Prerequisites:** Notebooks 43 (StoP ✓), 45 (InT ✓), 42 (DyS ✓)  

---

## Goal

Implement a **behavioral Statistical Mechanics analog** on DS5 by:

1. Deriving an effective behavioral inverse-temperature β_behav = 2/σ̂² from StoP diffusion estimates.
2. Implementing a two-state Boltzmann model: override (`high-use`) vs. non-override (`low-use`) states.
3. Computing behavioral susceptibility χ_behav (intervention sensitivity analog) per participant.
4. Estimating condition-level behavioral potential landscapes V(x) via KDE inversion (Level 2).
5. Producing a third entropy estimate H_StM to triangulate against H_SDE and H_emp from InT.
6. Closing the StoP ↔ DyS ↔ InT triangle into a **four-model quadrangle** via a convergence criterion.

## Not the goal

- No clustering, no phenotype labels, no personalization (Step 2 / notebook 100).
- No Ising coupling between participants (J = 0 throughout — DS5 participants are independent).
- No phase transition or critical temperature T_c estimation (only 3 discrete conditions).
- No hysteresis claims (DS5 has no post-intervention behavioral baseline).
- No claim that DS5 instantiates a full thermodynamic system at equilibrium.

## Core claim

> This notebook implements a **behavioral Statistical Mechanics approximation**. All thermodynamic quantities (β, T, V, χ, F) are behavioral analogs derived from the override-tendency proxy X_t. They are not physical quantities. The mapping β_behav = 2/σ̂² is an approximation — exact only for a potential-well SDE, not for the constant-drift SDE used in StoP. All claims are qualified accordingly.

## Output

```
outputs/model_outputs/statistical_mechanics/
  participant_stm_parameters.csv
  condition_stm_summary.csv
  condition_potential.csv
  stm_stop_int_dys_comparison.csv
  psychosocial_stm_correlations.csv
  model_run_summary.md
  figures/fig1_stm_beta_by_condition.png
  figures/fig2_stm_puse_chi_by_condition.png
  figures/fig3_stm_potential_landscape.png
  figures/fig4_stm_four_model_comparison.png
```


## FLAG decisions for DS5 StM Step 1

| Flag | Decision | Alternative (deferred) | Why deferred |
|---|---|---|---|
| FLAG 1 | Two-state behavioral Boltzmann model: X_t discretized into high-use (X_t > 0.5) and low-use (X_t ≤ 0.5); single-particle model (J = 0) | Continuous V(x) for individual participants | Individual-level V(x) requires dense P_inf; 7–45 observations too noisy |
| FLAG 2 | Effective temperature: β_behav = 2/σ̂² from StoP (constant-drift approximation) | OU-process β = 2θ/σ² with explicit mean-reversion rate θ | OU requires a third parameter θ; DS4 long windows are the correct target |
| FLAG 3 | No Ising coupling between participants: independent single-spin model per participant | Population-level Ising with participant interaction | DS5 participants are independent individuals; no interaction term available or justified |
| FLAG 4 | Phase transition analysis deferred: no critical temperature T_c estimated | T_c from continuous β sweep | Only 3 discrete conditions; cannot trace a phase boundary |


## ⚠ Proxy warning and StM-specific caveats

**This applies to every quantity computed in this notebook.**

```
X_t = n_opened_day / n_friction_events_day   ∈ [0, 1]
```

X_t approximates behavioral override tendency. It does **NOT** directly measure urge, self-control capacity, habit strength, attentional load, or any latent cognitive/psychological state.

### Additional StM-specific caveat: constant-drift SDE ≠ potential-well SDE

The Boltzmann bridge P_inf(x) ∝ exp(−2V(x)/σ²) is formally correct for the **potential-well SDE**: dX_t = −(dV/dx) dt + σ dW_t.  
The StoP notebook (43) used the **constant-drift SDE**: dX_t = μ dt + σ dW_t, which has V(x) = −μx (a linear slope, non-confining).  

**Consequence:** The mapping **β_behav = 2/σ̂²** is a behavioral analog / approximation, not a rigorous thermodynamic derivation. The quantitative values of β_behav are approximate; the qualitative ordering (higher σ̂ → higher T_behav → more behavioral exploration) is robust.  

This caveat is noted in every relevant cell as **FLAG A**.


## Theory 1 — Statistical Mechanics core (§4.1.1)

### Hamiltonian and energy landscape (Eq. 1)

$$H(q, p) = T(p) + V(q)$$

In the behavioral analog: V(q) is the potential energy of behavioral state q = X_t. T(p) (kinetic energy) has no direct behavioral equivalent and is suppressed (overdamped limit).

### Boltzmann distribution (Eq. 2)

$$P(s) = \frac{e^{-\beta H(s)}}{Z}$$

States with lower behavioral energy are more frequently occupied. β controls how sharply the distribution concentrates around the minimum-energy state.

### Partition function and free energy (Eqs. 3–4, 12–13)

$$Z = \sum_i e^{-E_i / k_B T}$$

$$F = -k_B T \ln Z$$

F = free energy; balances energy cost (E) against entropic exploration (S). Behavioral analog: lower F = habitual + predictable; higher F = flexible + exploratory.

### Thermodynamic entropy (Eq. 5)

$$S = k_B \ln \Omega$$

Bridge to InT: S ↔ H_emp (empirical behavioral entropy). H_StM from the Boltzmann-fitted distribution is the StM-specific prediction of S.

### Specific heat — behavioral fluctuations (Eq. 7)

$$C_v = \frac{\beta^2}{N} \left(\langle E^2 \rangle - \langle E \rangle^2\right)$$

Behavioral analog: C_behav = β² · Var(X_t) ≈ β² · σ̂². Under β = 2/σ̂² this is (2/σ̂²)²·σ̂² = 4/σ̂² = 2·β_behav — i.e. it is NOT constant; it is a redundant restatement of β_behav (verified: max|C_behav − 4/σ̂²| = 4.5e-13; observed range 7.8–4536, mean 229). Note also that Eq. 7's C_v = β²·Var(E) requires the variance of the STATE distribution, whereas σ̂² is the per-day INCREMENT variance from the SDE.

### Susceptibility (Eq. 8)

$$\chi = \left(\frac{\partial M}{\partial H}\right)_T$$

Behavioral analog: χ_behav = ∂⟨X_t⟩/∂h_friction. For the two-state model: χ_behav = β · p_use · (1 − p_use).


## Theory 2 — Ising model and external field (§4.1.1, Eqs. 10–11)

### Ising Hamiltonian (Eq. 10)

$$H = -J \sum_{\langle i,j \rangle} s_i s_j$$

J = spin-spin coupling constant. In DS5: **J = 0** (FLAG 3). Participants are independent individuals; no interaction term is available or justified.

### Ising with external field (Eq. 11)

$$H = -J \sum_{\langle i,j \rangle} s_i s_j - h \sum_i s_i$$

h = external field, here interpreted as friction condition strength.  

| DS5 condition | Friction field h | Interpretation |
|---|---|---|
| control | h ≈ 0 (reference) | No external behavioral field |
| default | h > 0 (friction present) | Field biases toward non-use state |
| reintervention | h > 0 (different field strength) | Field biases toward non-use, different intensity |

**Note:** h is ordinal, not cardinal. We cannot assign precise numerical h values from DS5 alone. Condition-level comparisons are the correct level of analysis.

### Single-particle Boltzmann at J = 0

With J = 0 and a single spin, the model reduces to:  
P(use) = e^{−β E_use} / Z₂   where Z₂ = e^{−β E_use} + e^{−β E_non-use}

This is the two-state model implemented in this notebook (FLAG 1).


## Theory 3 — Boltzmann bridge from StoP (FLAG A applies)

### Potential-well SDE and Boltzmann stationary distribution

For the **potential-well SDE**:  
$$dX_t = -\frac{dV}{dx} dt + \sigma \, dW_t$$

the stationary distribution is:  
$$P_\infty(x) \propto \exp\!\left(-\frac{2V(x)}{\sigma^2}\right)$$

Inverting: $V(x) = -\frac{\sigma^2}{2} \log P_\infty(x)$

Identifying β ≡ 2/σ²:  
$$P_\infty(x) \propto e^{-\beta V(x)} \quad \Leftrightarrow \quad \text{Boltzmann distribution}$$

### Effective behavioral temperature (⚠ FLAG A — approximation)

$$\beta_{\text{behav}} = \frac{2}{\hat{\sigma}^2}, \qquad T_{\text{behav}} = \frac{\hat{\sigma}^2}{2}$$

> **FLAG A caveat:** The StoP notebook used the constant-drift SDE dX_t = μ dt + σ dW_t,  
> which has V(x) = −μx (linear, non-confining). On the clipped [0,1] domain, the stationary  
> distribution concentrates at a boundary rather than forming a smooth Boltzmann density.  
> The mapping β_behav = 2/σ̂² is therefore an **approximation** — the qualitative ordering  
> (higher σ̂ → higher T_behav → more behavioral exploration) is robust, but the quantitative  
> values are approximate.

### Potential inversion from empirical data

$$\hat{V}(x) = -\frac{\bar{\sigma}^2_c}{2} \log \hat{P}_\infty(x)$$

where P̂_∞(x) is estimated via KDE on pooled daily X_t per condition (Level 2).  
Individual-level V(x) (Level 3) is flagged as exploratory due to sparse data (7–45 points).

### StM entropy bridge to InT

$$H_{\text{StM}} = -\int P_\infty(x) \log_2 P_\infty(x) \, dx$$

H_StM is the entropy of the Boltzmann-fitted distribution. It provides a **third entropy estimate** alongside H_SDE (parametric Gaussian from StoP) and H_emp (non-parametric from InT).  
Agreement across all three: convergent evidence for the Gaussian-Boltzmann approximation.  
> ⚠ None of H_SDE, H_StM, or the Boltzmann bridge equation currently appear in the paper — all must be added before submission.


## Theory 4 — Two-state behavioral Boltzmann model (FLAG 1)

### State discretization

| State | Definition | StM spin | Behavioral meaning |
|---|---|---|---|
| high-use | X_t > 0.5 | s = +1 | Override tendency dominant |
| low-use | X_t ≤ 0.5 | s = −1 | Friction compliance dominant |

### Behavioral energies

Using the information-theoretic energy (negative log-probability = surprisal):  
$$E_{\text{use}} = -\ln \hat{p}_{\text{use}}, \qquad E_{\text{non-use}} = -\ln(1 - \hat{p}_{\text{use}})$$

> **FLAG G:** This 'energy' is information-theoretic (surprisal), not physical energy. States with lower surprisal (higher empirical probability) are assigned lower energy.

> **FLAG G2 (identification caveat):** Because E_use := −ln p̂_use, the two-state Boltzmann
> probability is P(use) = p̂^β / (p̂^β + (1−p̂)^β), which reproduces the empirical p̂ **only at
> β = 1**. With the fitted β (median 61, max 2268) the model saturates: 377/414 participants
> (91%) have P(use) > 0.999 or < 0.001, and Z_2state underflows (min 2.3e-266; 136/414 below
> 1e-10). **Z_2state must therefore not be used for F = −kT ln Z**, and the exported
> χ_behav = β·p̂·(1−p̂) is the β=1 susceptibility evaluated at the empirical p̂ — it is NOT the
> susceptibility of the exported partition function (that would be ≈ 4·β·P·(1−P) ≈ 0 for the
> same 91% of participants). Z_2state is a diagnostic-only column; it is deliberately excluded
> from phenotyping (see ds5_1st_phenotyping_blueprint.md).

### Two-state partition function (not yet in paper — must add)

$$Z_2 = e^{-\beta E_{\text{use}}} + e^{-\beta E_{\text{non-use}}}$$

### Behavioral susceptibility (not yet in paper — must add)

$$\chi_{\text{behav}} = \beta \cdot \hat{p}_{\text{use}} \cdot (1 - \hat{p}_{\text{use}})$$

Highest χ for participants near p_use ≈ 0.5 with high β: closest to a 'tipping point' between override and compliance.

### Behavioral free energy analog (not yet in paper — must add)

$$F_{\text{behav}} = E_{\text{use}} - \frac{1}{\beta} \cdot H_{\text{emp}}$$

Low F_behav: habitual (low energy) + predictable (low entropy) — 'locked-in' profile.  
High F_behav: high-energy + high-entropy — 'exploratory / susceptible' profile.


## Theory-to-data mapping table

| Theoretical quantity | DS5 operationalization | Confidence | Key caveat |
|---|---|---|---|
| Microstate s | X_t = n_opened_day / n_events_day (continuous) or {high-use, low-use} (two-state) | Medium | Behavioral proxy; not latent urge or self-control |
| Temperature T | T_behav = σ̂²/2 per participant | Medium | ⚠ Approximation (FLAG A): exact only for potential-well SDE |
| Inverse temperature β | β_behav = 2/σ̂² per participant | Medium | ⚠ Approximation (FLAG A) |
| Potential V(x) | V̂(x) = −(σ̄²/2) log KDE(x) per condition | Low | ⚠ Condition-level only (FLAG B); individual V(x) too noisy |
| Partition function Z | Z_2 = e^{−βE_use} + e^{−βE_non-use} per participant | Low–Medium | Two-state simplification (FLAG 1) |
| Free energy F | F_behav = E_use − (1/β) · H_emp | Low–Medium | Behavioral analog; not physically derived |
| Entropy S | H_emp from InT (empirical); H_StM from Boltzmann fit | Medium | Three-way comparison: H_SDE ↔ H_StM ↔ H_emp |
| Susceptibility χ | χ_behav = β · p_use · (1 − p_use) | Low–Medium | Condition-level comparison most defensible |
| External field h | Friction condition (control < default ≈ reintervention) | Low–Medium | Ordinal only; no cardinal h values |
| Magnetization ⟨M⟩ | x_mean per participant (from StoP) | Medium | Well-defined behavioral quantity |
| Energy minimum | Peak of KDE P̂_inf(x) per condition | Low | ⚠ Noisy for individuals; condition-level only |
| Specific heat C_v | β² · σ̂² = 4/σ̂² = 2·β_behav under β = 2/σ̂² | Low | NOT constant; exactly 2× β_behav, so carries no information beyond β; also uses increment variance, not state variance |
| Ising coupling J | J = 0 (FLAG 3) | N/A | No interaction between participants in DS5 |
| Phase transition T_c | NOT AVAILABLE | None | Only 3 discrete conditions (FLAG E) |
| Hysteresis | NOT AVAILABLE | None | No post-intervention behavioral baseline (FLAG F) |


## Equation gaps — what must be added to the paper before submission

| Equation | Formula | Paper status | Required action |
|---|---|---|---|
| Boltzmann bridge (StoP → StM) | P_inf(x) ∝ exp(−2V(x)/σ²) | ❌ Not in paper | Add to §4.1.1 or §4.1.3; include FLAG A caveat about constant-drift |
| Effective behavioral temperature | β_behav = 2/σ̂² (approximation) | ❌ Not in paper | Add to §4.2.1; explicitly flag as approximation for constant-drift SDE |
| Potential inversion | V(x) = −(σ̂²/2) log P̂_inf(x) | ❌ Not in paper | Add to §4.2.1; note condition-level vs individual-level stability |
| Two-state partition function | Z_2 = e^{−βE_use} + e^{−βE_non-use} | ❌ Not in paper | Add to §4.2.1 — simplest defensible DS5 implementation |
| Behavioral susceptibility | χ_behav = β · p_use · (1 − p_use) | ❌ Not in paper | Add to §4.2.1 — intervention sensitivity measure |
| StM entropy bridge | H_StM = −∫ P_inf log₂ P_inf dx | ❌ Not in paper | Add to §4.2.1 or §5 — needed for three-way entropy triangulation |
| Behavioral free energy | F_behav = E_behav − (1/β) · S_behav | ❌ Not in paper | Add as conceptual equation §4.2.1 |
| Constant-drift ≠ potential-well SDE | Explicit caveat | ❌ Not in paper | Add to §4.2.1 limitations — critical for scientific accuracy |


In [ ]:
# Cell 10 — Setup
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
from scipy.integrate import simpson
from pathlib import Path
from datetime import datetime

# ── Paths ────────────────────────────────────────────────────────────────
ROOT = Path('.').resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

OUT_DIR   = ROOT / 'outputs' / 'model_outputs' / 'statistical_mechanics'
FIG_DIR   = OUT_DIR / 'figures'
STOP_DIR  = ROOT / 'outputs' / 'model_outputs' / 'stochastic_processes'
INT_DIR   = ROOT / 'outputs' / 'model_outputs' / 'information_theory'
DYS_DIR   = ROOT / 'outputs' / 'model_outputs' / 'dynamical_systems'
CLEAN_DIR = ROOT / 'outputs' / 'cleaned'
FEAT_DIR  = ROOT / 'outputs' / 'features' / 'ds5'

OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

# ── Model constants ───────────────────────────────────────────────────────
MIN_DAYS        = 7           # inherited from StoP
THRESHOLD_USE   = 0.5         # X_t > 0.5 → high-use state
KDE_BANDWIDTH   = 'silverman' # scipy gaussian_kde bandwidth
KDE_GRID_N      = 200         # number of grid points for V(x)
EPS             = 1e-9        # numerical stability
PSYCH_MIN_N     = 20          # min n for psychosocial correlations
MU_THRESHOLD    = 0.005       # boundary fp guard (from StoP)

# ── Sigma/H_emp thresholds (from DyS/InT) ────────────────────────────────
SIGMA_THRESHOLD = 0.181220    # DyS sigma->0 partition: median sigma_hat

print(f'Output directory: {OUT_DIR}')
print(f'Figures directory: {FIG_DIR}')
print(f'Run time: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')


In [ ]:
# Cell 11 — Data loading
# Load StoP, InT, DyS outputs — no new data pipeline required for Level 1

# StoP participant parameters
df_stop = pd.read_csv(STOP_DIR / 'participant_parameters.csv')
print(f'StoP: {len(df_stop)} rows')

# InT entropy metrics
df_int = pd.read_csv(INT_DIR / 'participant_entropy_metrics.csv')
print(f'InT:  {len(df_int)} rows')

# DyS dynamics parameters
df_dys = pd.read_csv(DYS_DIR / 'participant_dynamics_parameters.csv')
print(f'DyS:  {len(df_dys)} rows')

# Survey features (psychosocial scales)
df_surv = pd.read_csv(FEAT_DIR / 'ds5_survey_features.csv')
print(f'Survey: {len(df_surv)} rows')

# ── Merge all three model outputs on participant_id_canonical ─────────────
# Use StoP as left spine (414 eligible participants)
df = df_stop[['participant_id_canonical','study_condition','n_days',
              'mu_hat','sig_hat','x_mean','p_high_days','p_high_days_smoothed',
              'sig_floor','sig_floor_ratio','sig_excess']].copy()

# Join InT columns (H_emp, H_SDE)
int_cols = ['participant_id_canonical','H_emp','H_SDE']
df = df.merge(df_int[int_cols], on='participant_id_canonical', how='left')

# Join DyS columns (is_dys_eligible, behaviorally_deterministic)
dys_cols = ['participant_id_canonical','is_dys_eligible','behaviorally_deterministic',
            'predicted_boundary_fp']
df = df.merge(df_dys[dys_cols], on='participant_id_canonical', how='left')

print(f'\nMerged: {len(df)} participants')
print(f'Conditions: {df["study_condition"].value_counts().to_dict()}')
print(f'NaN H_emp:  {df["H_emp"].isna().sum()}')
print(f'NaN is_dys: {df["is_dys_eligible"].isna().sum()}')


In [ ]:
# Cell 12 — Level 1: Effective behavioral temperature β_behav = 2/σ̂²
# FLAG A: approximation — exact only for potential-well SDE

df['beta_behav'] = 2.0 / df['sig_hat']**2
df['T_behav']    = df['sig_hat']**2 / 2.0

# Sanity check: β ordering should be control > reintervention > default
# (because σ̂ default > reintervention > control)
beta_by_cond = df.groupby('study_condition')['beta_behav'].mean().sort_values(ascending=False)
print('Mean β_behav by condition (expected order: control > reintervention > default):')
print(beta_by_cond.round(2))

T_by_cond = df.groupby('study_condition')['T_behav'].mean().sort_values()
print('\nMean T_behav by condition (expected order: control < reintervention < default):')
print(T_by_cond.round(4))

print(f'\nβ_behav: min={df["beta_behav"].min():.1f}, median={df["beta_behav"].median():.1f}, max={df["beta_behav"].max():.1f}')


In [ ]:
# Cell 13 — Level 1: Two-state model, susceptibility χ_behav, free energy F_behav

# Two-state high-use probability, matching the declared definition P(X_t > 0.5).
# Add-one smoothing avoids infinite surprisal for short series at exactly 0 or 1.
df['p_use_approx'] = df['p_high_days_smoothed']

# Behavioral energies (information-theoretic: surprisal = -ln p)
# FLAG G: 'energy' here is informational (negative log-probability), not physical energy
df['E_use']    = -np.log(df['p_use_approx'])
df['E_nonuse'] = -np.log(1.0 - df['p_use_approx'])

# Two-state partition function (FLAG 1, not yet in paper)
df['Z_2state'] = (np.exp(-df['beta_behav'] * df['E_use']) +
                  np.exp(-df['beta_behav'] * df['E_nonuse']))

# Behavioral susceptibility (not yet in paper)
df['chi_behav'] = df['beta_behav'] * df['p_use_approx'] * (1.0 - df['p_use_approx'])

# Behavioral free energy analog (not yet in paper)
# F_behav = E_use - (1/β) * H_emp
# H_emp is in bits; E_use is in nats, so convert H_emp before combining them.
# Convert H_emp from bits to nats before combining it with E_use (natural-log surprisal).
H_emp_nats = df['H_emp'].fillna(df['H_emp'].median()) * np.log(2.0)
df['F_behav'] = df['E_use'] - (1.0 / df['beta_behav']) * H_emp_nats

# Behavioral specific heat analog
# C_behav = β² · σ̂² = (2/σ̂²)² · σ̂² = 4/σ̂² = 2·β_behav  (NOT constant; redundant with β_behav)
df['C_behav'] = df['beta_behav']**2 * df['sig_hat']**2

print('Two-state model quantities by condition:')
cols = ['p_use_approx','E_use','chi_behav','F_behav','beta_behav']
print(df.groupby('study_condition')[cols].mean().round(4))

print(f'\nC_behav (= 4/σ̂² = 2·β_behav): mean={df["C_behav"].mean():.3f}, std={df["C_behav"].std():.3f}')
print('(Exactly 2× β_behav — redundant with β_behav, no independent discriminative value)')


In [ ]:
# Cell 14 — Level 2: Condition-level V(x) from KDE inversion
# Pools all daily X_t values per condition (~3,000-4,000 points each)
# FLAG B: condition-level is more stable; individual-level is too noisy (7-45 points)

# Load behavioral clean file and join study_condition
df_beh = pd.read_csv(CLEAN_DIR / 'ds5_behavioral_clean.csv',
                     usecols=['participant_id_canonical','event_timestamp_utc',
                              'event_outcome_simple'])

# Join study_condition from survey features
cond_map = df_surv.set_index('participant_id_canonical')['study_condition'].to_dict()
df_beh['study_condition'] = df_beh['participant_id_canonical'].map(cond_map)

# Filter valid outcomes (same as StoP)
valid_outcomes = {'opened', 'dismissed'}
df_beh = df_beh[df_beh['event_outcome_simple'].isin(valid_outcomes)].copy()

# Parse date
df_beh['date'] = pd.to_datetime(df_beh['event_timestamp_utc'], utc=True).dt.date

# Daily X_t per participant
daily = (df_beh.groupby(['participant_id_canonical','date','study_condition'])
         .agg(n_opened=('event_outcome_simple', lambda x: (x=='opened').sum()),
              n_events=('event_outcome_simple', 'count'))
         .reset_index())
daily['X_t'] = daily['n_opened'] / daily['n_events']

# Restrict to MIN_DAYS-eligible participants (use StoP eligible set)
eligible_ids = set(df['participant_id_canonical'])
daily_elig = daily[daily['participant_id_canonical'].isin(eligible_ids)].copy()

# Pool X_t by condition for KDE
x_grid = np.linspace(0.001, 0.999, KDE_GRID_N)
condition_potential = []

for cond in ['control','default','reintervention']:
    x_vals = daily_elig.loc[daily_elig['study_condition']==cond, 'X_t'].values
    x_vals = x_vals[np.isfinite(x_vals)]
    sig_bar = df.loc[df['study_condition']==cond, 'sig_hat'].mean()
    n_points = len(x_vals)

    # Reflection-corrected KDE on the bounded [0,1] domain. Pin the absolute
    # bandwidth to the unreflected sample so tripling the sample cannot inflate it.
    kde_raw = stats.gaussian_kde(x_vals, bw_method=KDE_BANDWIDTH)
    bw_abs = kde_raw.factor * np.std(x_vals, ddof=1)
    x_refl = np.concatenate([-x_vals, x_vals, 2.0 - x_vals])
    kde = stats.gaussian_kde(x_refl, bw_method=bw_abs / np.std(x_refl, ddof=1))
    kde_density = kde(x_grid) * 3.0
    kde_density = np.maximum(kde_density, EPS)
    kde_density /= np.trapz(kde_density, x_grid)  # renormalize

    # V(x) = -(σ̄²/2) log P̂_inf(x) — behavioral potential analog
    V_x = -(sig_bar**2 / 2.0) * np.log(kde_density + EPS)
    V_x -= V_x.min()  # shift minimum to 0 for visualization

    # H_StM (condition-level): entropy of the Boltzmann-fitted distribution
    H_StM_cond = -np.sum(kde_density * np.log2(kde_density + EPS)) * (x_grid[1] - x_grid[0])

    # Location and depth of potential minimum (= P_inf peak)
    V_min_idx    = np.argmin(V_x)
    V_min_loc    = x_grid[V_min_idx]
    V_min_depth  = V_x.max() - V_x.min()

    print(f'[{cond:16s}] n={n_points:5d}  σ̄={sig_bar:.4f}  '
          f'V_min_loc={V_min_loc:.3f}  H_StM={H_StM_cond:.3f} bits')

    for xi, kde_i, V_i in zip(x_grid, kde_density, V_x):
        condition_potential.append(dict(
            study_condition=cond, x_grid=xi,
            kde_density=kde_i, V_x=V_i,
            n_pooled=n_points, H_StM_condition=H_StM_cond,
            V_min_location=V_min_loc, V_min_depth=V_min_depth,
        ))

df_pot = pd.DataFrame(condition_potential)
print(f'\nCondition potential table: {len(df_pot)} rows')


In [ ]:
# Cell 15 — Compare V(x) minima with condition-level x_mean from StoP

v_summary = (df_pot.groupby('study_condition')
             .agg(V_min_location=('V_min_location','first'),
                  V_min_depth=('V_min_depth','first'),
                  H_StM_condition=('H_StM_condition','first'))
             .reset_index())

x_mean_cond = df.groupby('study_condition')['x_mean'].mean().reset_index()
x_mean_cond.columns = ['study_condition','x_mean_StoP']

v_compare = v_summary.merge(x_mean_cond, on='study_condition')
v_compare['delta_Vmin_xmean'] = (v_compare['V_min_location'] - v_compare['x_mean_StoP']).abs()

print('V(x) minimum location vs StoP x_mean:')
print(v_compare[['study_condition','V_min_location','x_mean_StoP',
                  'delta_Vmin_xmean','H_StM_condition']].to_string(index=False))
print('\n(V_min is expected to be boundary-pinned after reflection correction because'
      '\n many participant-days have X_t exactly 1; it is not a condition contrast.)')


In [ ]:
# Cell 16 — H_StM: three-entropy triangulation (H_SDE ↔ H_StM ↔ H_emp)

# Condition-level means from InT
h_summary = df.groupby('study_condition').agg(
    H_emp_mean=('H_emp','mean'),
    H_SDE_mean=('H_SDE','mean'),
).reset_index()

h_summary = h_summary.merge(
    v_summary[['study_condition','H_StM_condition']], on='study_condition')

print('Three-entropy comparison by condition:')
print('(consistent ordering = convergent evidence for Gaussian-Boltzmann approximation)')
print(h_summary[['study_condition','H_SDE_mean','H_StM_condition','H_emp_mean']].to_string(index=False))

# Cross-model correlation: H_StM (participant-level approx) vs H_emp
# Approximate individual H_StM using same KDE approach on each participant's X_t series
# For now, compute from β: H_StM_participant = 0.5*log2(2*pi*e / beta_behav)
# (This is the differential entropy of N(0, T_behav) = N(0, σ²/2))
# Note: this is the same as H_SDE with σ replaced by σ/√2 — flagged as approximate
df['H_StM_participant'] = 0.5 * np.log2(2 * np.pi * np.e * df['T_behav'])

r_stm_emp, p_stm_emp = stats.pearsonr(
    df['H_StM_participant'].dropna(),
    df.loc[df['H_StM_participant'].notna(), 'H_emp']
)
r_sde_emp, p_sde_emp = stats.pearsonr(
    df['H_SDE'].dropna(),
    df.loc[df['H_SDE'].notna(), 'H_emp']
)
print(f'\nCross-model entropy correlations (participant-level):')
print(f'  r(H_SDE, H_emp)  = {r_sde_emp:.3f}  p={p_sde_emp:.4f}  (from InT — benchmark)')
print(f'  r(H_StM, H_emp)  = {r_stm_emp:.3f}  p={p_stm_emp:.4f}  (new — StM bridge)')
print(f'  Note: H_StM_participant = f(σ̂/√2) — closely related to H_SDE = f(σ̂)')
print(f'  High r(H_StM, H_SDE) is expected — both are monotone transforms of σ̂')


In [ ]:
# Cell 17 — Susceptibility analysis

# Condition-level behavioral susceptibility
chi_cond = df.groupby('study_condition').agg(
    n=('chi_behav','count'),
    mean_chi=('chi_behav','mean'),
    median_chi=('chi_behav','median'),
    mean_puse=('p_use_approx','mean'),
    mean_beta=('beta_behav','mean'),
).reset_index()

print('Susceptibility by condition:')
print(chi_cond.to_string(index=False))

# Identify high-χ participants (top quartile)
chi_q75 = df['chi_behav'].quantile(0.75)
df['high_chi'] = df['chi_behav'] > chi_q75
print(f'\nHigh-χ participants (top 25%): n={df["high_chi"].sum()}')
print(df.groupby('study_condition')['high_chi'].mean().round(3))

# Condition-level susceptibility using observed x_mean differences
# χ_cond = Δ⟨X_t⟩ / (condition difference as a proxy for Δh)
x_ctrl = df.loc[df['study_condition']=='control', 'x_mean'].mean()
x_def  = df.loc[df['study_condition']=='default', 'x_mean'].mean()
x_rei  = df.loc[df['study_condition']=='reintervention', 'x_mean'].mean()

print(f'\nObserved mean x_mean: control={x_ctrl:.4f}, default={x_def:.4f}, reintervention={x_rei:.4f}')
print(f'Δ⟨X_t⟩ (ctrl-default)={x_ctrl-x_def:.4f}, (ctrl-reintervention)={x_ctrl-x_rei:.4f}')
print('(larger Δ⟨X_t⟩ for default is consistent with it having lower β / higher T_behav)')


In [ ]:
# Cell 18 — Free energy analysis: locked-in vs exploratory behavioral regimes

F_cond = df.groupby('study_condition').agg(
    mean_F=('F_behav','mean'),
    median_F=('F_behav','median'),
    std_F=('F_behav','std'),
).reset_index()

print('Behavioral free energy by condition:')
print(F_cond.to_string(index=False))

# 'Locked-in' profile: low E_use + low H_emp → low F_behav
# 'Exploratory' profile: high E_use + high H_emp → high F_behav
F_median = df['F_behav'].median()
df['behavioral_regime'] = np.where(df['F_behav'] < F_median, 'locked_in', 'exploratory')

print(f'\nBehavioral regime split (F_median={F_median:.3f}):')
regime_cond = df.groupby(['study_condition','behavioral_regime']).size().unstack(fill_value=0)
print(regime_cond)
print('\nFraction locked-in by condition:')
print((regime_cond['locked_in'] / regime_cond.sum(axis=1)).round(3))


In [ ]:
# Cell 19 — Four-model convergence: StoP + DyS + InT + StM

# Convergence criterion:
#   low σ̂ (StoP)           → sig_hat < SIGMA_THRESHOLD
#   DyS-eligible            → is_dys_eligible == True
#   low H_emp (InT)         → H_emp < H_emp_median
#   high β_behav (StM)      → beta_behav > beta_median

H_emp_median = df['H_emp'].median()
beta_median  = df['beta_behav'].median()

df['four_model_convergent'] = (
    (df['sig_hat'] < SIGMA_THRESHOLD) &
    (df['is_dys_eligible'] == True) &
    (df['H_emp'] < H_emp_median) &
    (df['beta_behav'] > beta_median)
)

n_conv = df['four_model_convergent'].sum()
pct_conv = n_conv / len(df) * 100
print(f'Four-model convergent participants: n={n_conv} ({pct_conv:.1f}%)')
print('(These participants have low noise, deterministic dynamics, low entropy, and high β)')
print('(= thermodynamically ordered, behaviorally entrenched profile)')

print('\nConvergent participants by condition:')
print(df.groupby('study_condition')['four_model_convergent'].agg(['sum','mean']).round(3))

# Check overlap with is_dys_eligible (expected: strong, both derive from σ̂)
overlap = (df['four_model_convergent'] & df['is_dys_eligible']).sum()
print(f'\nOverlap: four_model_convergent ∩ is_dys_eligible = {overlap}')
print(f'(All four_model_convergent must be DyS-eligible by definition)')

# Summary statistics for convergent vs non-convergent
key_cols = ['sig_hat','beta_behav','H_emp','chi_behav','F_behav','x_mean']
print('\nMean values — convergent vs non-convergent:')
print(df.groupby('four_model_convergent')[key_cols].mean().round(4))


In [ ]:
# Cell 20 — Psychosocial correlates (exploratory, friction-exposed only)

# Merge psychosocial scales
psych_cols = ['participant_id_canonical','fomo_mean_pre','ostracism_mean_pre',
              'digital_wellbeing_pre','social_media_wellbeing_pre']
psych_cols_avail = [c for c in psych_cols if c in df_surv.columns]
df_psych = df.merge(df_surv[psych_cols_avail], on='participant_id_canonical', how='left')

# Restrict to friction-exposed (default + reintervention)
df_friction = df_psych[df_psych['study_condition'].isin(['default','reintervention'])].copy()
print(f'Friction-exposed for psychosocial analysis: n={len(df_friction)}')

stm_features = ['beta_behav','T_behav','chi_behav','F_behav']
psych_scales  = [c for c in psych_cols_avail if c != 'participant_id_canonical']

results = []
for feat in stm_features:
    for scale in psych_scales:
        sub = df_friction[[feat, scale]].dropna()
        if len(sub) < PSYCH_MIN_N:
            continue
        r, p = stats.pearsonr(sub[feat], sub[scale])
        results.append(dict(stm_feature=feat, psych_scale=scale,
                            r=r, p_value=p, n=len(sub)))

df_corr = pd.DataFrame(results)
if len(df_corr) > 0:
    df_corr_sig = df_corr[df_corr['p_value'] < 0.05].sort_values('p_value')
    print(f'\nNominally significant (p < 0.05): {len(df_corr_sig)} / {len(df_corr)} tests')
    if len(df_corr_sig) > 0:
        print(df_corr_sig[['stm_feature','psych_scale','r','p_value','n']].to_string(index=False))
    else:
        print('None nominally significant.')
    print(f'\n⚠ {len(df_corr)} total tests — no multiple-comparison correction (exploratory only)')
else:
    print('No psychosocial correlations computed (insufficient sample sizes).')
    df_corr = pd.DataFrame(columns=['stm_feature','psych_scale','r','p_value','n'])


In [ ]:
# Cell 21 — Export all CSV outputs

# ── 1. participant_stm_parameters.csv ────────────────────────────────────
participant_cols = [
    'participant_id_canonical','study_condition','n_days',
    'mu_hat','sig_hat','x_mean',
    'beta_behav','T_behav','p_use_approx',
    'E_use','E_nonuse','Z_2state','chi_behav','F_behav','C_behav',
    'H_StM_participant','H_emp','H_SDE',
    'is_dys_eligible','behaviorally_deterministic',
    'high_chi','behavioral_regime','four_model_convergent',
]
participant_cols = [c for c in participant_cols if c in df.columns]
df[participant_cols].to_csv(OUT_DIR / 'participant_stm_parameters.csv', index=False)
print(f'Saved participant_stm_parameters.csv ({len(df)} rows)')

# ── 2. condition_stm_summary.csv ─────────────────────────────────────────
cond_summary = df.groupby('study_condition').agg(
    n=('participant_id_canonical','count'),
    mean_sig_hat=('sig_hat','mean'),
    mean_beta=('beta_behav','mean'),
    median_beta=('beta_behav','median'),
    mean_T_behav=('T_behav','mean'),
    mean_p_use=('p_use_approx','mean'),
    mean_chi=('chi_behav','mean'),
    median_chi=('chi_behav','median'),
    mean_F_behav=('F_behav','mean'),
    mean_H_emp=('H_emp','mean'),
    mean_H_SDE=('H_SDE','mean'),
    n_four_model_convergent=('four_model_convergent','sum'),
    frac_four_model_convergent=('four_model_convergent','mean'),
).reset_index()

# Add H_StM and V_min from condition potential table
cond_summary = cond_summary.merge(
    v_summary[['study_condition','H_StM_condition','V_min_location','V_min_depth']],
    on='study_condition', how='left')

cond_summary.to_csv(OUT_DIR / 'condition_stm_summary.csv', index=False)
print(f'Saved condition_stm_summary.csv ({len(cond_summary)} rows)')

# ── 3. condition_potential.csv ────────────────────────────────────────────
df_pot.to_csv(OUT_DIR / 'condition_potential.csv', index=False)
print(f'Saved condition_potential.csv ({len(df_pot)} rows)')

# ── 4. stm_stop_int_dys_comparison.csv ───────────────────────────────────
compare_cols = [
    'participant_id_canonical','study_condition',
    'mu_hat','sig_hat','x_mean',
    'H_emp','H_SDE','H_StM_participant',
    'is_dys_eligible','behaviorally_deterministic',
    'beta_behav','T_behav','chi_behav','F_behav',
    'four_model_convergent',
]
compare_cols = [c for c in compare_cols if c in df.columns]
df[compare_cols].to_csv(OUT_DIR / 'stm_stop_int_dys_comparison.csv', index=False)
print(f'Saved stm_stop_int_dys_comparison.csv ({len(df)} rows)')

# ── 5. psychosocial_stm_correlations.csv ──────────────────────────────────
df_corr.to_csv(OUT_DIR / 'psychosocial_stm_correlations.csv', index=False)
print(f'Saved psychosocial_stm_correlations.csv ({len(df_corr)} rows)')


In [ ]:
# Cell 22 — Figure 1: β_behav distribution by condition
# Mirrors StoP fig1 (violin plot structure)
# FLAG A caveat noted in subtitle

CONDITIONS = ['control','default','reintervention']
COND_COLORS = {'control':'#4878CF','default':'#D65F5F','reintervention':'#6ACC65'}

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('DS5 StM — Effective Behavioral Temperature\n'
             r'(β_behav = 2/σ̂²  ⚠ approximation: constant-drift SDE, not potential-well SDE)',
             fontsize=11)

for ax, col, label in zip(axes, ['beta_behav','T_behav'],
                          [r'β$_{behav}$ = 2/σ̂²', r'T$_{behav}$ = σ̂²/2']):
    data = [df.loc[df['study_condition']==c, col].dropna().values for c in CONDITIONS]
    vp = ax.violinplot(data, positions=[1,2,3], showmedians=True, widths=0.7)
    for body, cond in zip(vp['bodies'], CONDITIONS):
        body.set_facecolor(COND_COLORS[cond])
        body.set_alpha(0.6)
    for part in ['cmedians','cbars','cmins','cmaxes']:
        vp[part].set_color('black')
        vp[part].set_linewidth(1.2)
    ax.set_xticks([1,2,3])
    ax.set_xticklabels(CONDITIONS)
    ax.set_ylabel(label)
    ax.set_title(label)
    # Add condition means
    for pos, c in enumerate(CONDITIONS, 1):
        m = df.loc[df['study_condition']==c, col].mean()
        ax.axhline(m, xmin=(pos-1.3)/3, xmax=(pos-0.7)/3,
                   color=COND_COLORS[c], linewidth=2, linestyle='--', alpha=0.9)

plt.tight_layout()
fig.savefig(FIG_DIR / 'fig1_stm_beta_by_condition.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print('Saved fig1_stm_beta_by_condition.png')


In [ ]:
# Cell 23 — Figure 2: Two-state model — p_use and χ_behav by condition

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('DS5 StM — Two-State Boltzmann Model (FLAG 1: X_t discretized to high/low-use)',
             fontsize=11)

for ax, col, label in zip(axes,
                          ['p_use_approx', 'chi_behav'],
                          ['p_use = smoothed P(X_t > 0.5)', r'χ$_{behav}$ = β · p · (1−p)']):
    data = [df.loc[df['study_condition']==c, col].dropna().values for c in CONDITIONS]
    bp = ax.boxplot(data, positions=[1,2,3], patch_artist=True,
                   medianprops=dict(color='black',linewidth=2), widths=0.5)
    for patch, cond in zip(bp['boxes'], CONDITIONS):
        patch.set_facecolor(COND_COLORS[cond])
        patch.set_alpha(0.7)
    ax.set_xticks([1,2,3])
    ax.set_xticklabels(CONDITIONS)
    ax.set_ylabel(label)
    ax.set_title(label)

plt.tight_layout()
fig.savefig(FIG_DIR / 'fig2_stm_puse_chi_by_condition.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print('Saved fig2_stm_puse_chi_by_condition.png')


In [ ]:
# Cell 24 — Figure 3: Condition-level V(x) energy landscape

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('DS5 StM — Behavioral Potential Landscape V(x) (Level 2, condition-level KDE)\n'
             r'V̂(x) = −(σ̄²/2) log P̂$_∞$(x)  ⚠ approximate: pooled KDE, not individual participants',
             fontsize=10)

# Left panel: KDE density
ax = axes[0]
for cond in CONDITIONS:
    sub = df_pot[df_pot['study_condition']==cond]
    ax.plot(sub['x_grid'].values, sub['kde_density'].values, color=COND_COLORS[cond],
            linewidth=2, label=cond)
ax.set_xlabel('X_t (daily override proportion)')
ax.set_ylabel(r'$\hat{P}_\infty(x)$ — estimated stationary density')
ax.set_title('Estimated behavioral state distribution')
ax.legend()

# Right panel: V(x) potential
ax = axes[1]
for cond in CONDITIONS:
    sub = df_pot[df_pot['study_condition']==cond]
    ax.plot(sub['x_grid'].values, sub['V_x'].values, color=COND_COLORS[cond],
            linewidth=2, label=cond)
    # Mark V minimum
    v_min_loc = float(sub.loc[sub['V_x'].idxmin(), 'x_grid'])
    v_min_val = float(sub['V_x'].min())
    ax.scatter([v_min_loc], [v_min_val], color=COND_COLORS[cond], s=80, zorder=5)
ax.set_xlabel('X_t (daily override proportion)')
ax.set_ylabel(r'$\hat{V}(x)$ — behavioral potential (shifted to min=0)')
ax.set_title('Behavioral potential landscape (dots = V minima)')
ax.legend()

plt.tight_layout()
fig.savefig(FIG_DIR / 'fig3_stm_potential_landscape.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print('Saved fig3_stm_potential_landscape.png')


In [ ]:
# Cell 25 — Figure 4: Four-model comparison
# β_behav vs H_emp scatter; colored by is_dys_eligible; size-coded by χ_behav

fig, ax = plt.subplots(figsize=(9, 7))
fig.suptitle('DS5 StM — Four-Model Behavioral Profile (StoP + DyS + InT + StM)\n'
             r'β$_{behav}$ vs H$_{emp}$; color = DyS-eligible; size ∝ χ$_{behav}$',
             fontsize=10)

for is_dys, marker, label in [(True, 'o', 'DyS-eligible (low σ)'),
                               (False, 's', 'StoP-stochastic (high σ)')]:
    sub = df[df['is_dys_eligible'] == is_dys]
    scatter_color = [COND_COLORS[c] for c in sub['study_condition']]
    chi_norm = (sub['chi_behav'] - df['chi_behav'].min()) / (df['chi_behav'].max() - df['chi_behav'].min())
    sizes = 20 + 120 * chi_norm.values
    ax.scatter(sub['beta_behav'], sub['H_emp'],
               c=scatter_color, s=sizes, alpha=0.55,
               marker=marker, edgecolors='none', label=label)

# Legend for conditions
patches = [mpatches.Patch(color=COND_COLORS[c], label=c) for c in CONDITIONS]
patches += [plt.Line2D([0],[0], marker='o', color='gray', markersize=8, label='DyS-eligible'),
            plt.Line2D([0],[0], marker='s', color='gray', markersize=8, label='StoP-stochastic')]
ax.legend(handles=patches, fontsize=8, loc='upper right')

ax.set_xlabel(r'β$_{behav}$ = 2/σ̂²  (higher β = lower behavioral temperature, more ordered)')
ax.set_ylabel(r'H$_{emp}$ (empirical behavioral entropy, bits)')
ax.set_title('Four-model convergent region: high β ∩ low H_emp ∩ DyS-eligible (lower-right, circles)')

# Mark four-model convergent region
beta_med = df['beta_behav'].median()
Hemp_med = df['H_emp'].median()
ax.axvline(beta_med, color='gray', linewidth=1, linestyle=':', alpha=0.7, label=f'β median={beta_med:.1f}')
ax.axhline(Hemp_med, color='gray', linewidth=1, linestyle=':', alpha=0.7, label=f'H_emp median={Hemp_med:.2f}')
ax.text(beta_med*1.02, Hemp_med*0.95, 'four-model\nconvergent', fontsize=8, color='gray')

plt.tight_layout()
fig.savefig(FIG_DIR / 'fig4_stm_four_model_comparison.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print('Saved fig4_stm_four_model_comparison.png')


In [ ]:
# Cell 26 — Write model_run_summary.md

beta_ctrl = df.loc[df['study_condition']=='control','beta_behav'].mean()
beta_def  = df.loc[df['study_condition']=='default','beta_behav'].mean()
beta_rei  = df.loc[df['study_condition']=='reintervention','beta_behav'].mean()
chi_ctrl  = df.loc[df['study_condition']=='control','chi_behav'].mean()
chi_def   = df.loc[df['study_condition']=='default','chi_behav'].mean()
chi_rei   = df.loc[df['study_condition']=='reintervention','chi_behav'].mean()
n_conv    = int(df['four_model_convergent'].sum())
pct_conv  = n_conv / len(df) * 100

# Nominal psychosocial findings
psych_note = 'None nominally significant.'
if len(df_corr) > 0:
    sig_rows = df_corr[df_corr['p_value'] < 0.05]
    if len(sig_rows) > 0:
        psych_note = '; '.join([
            f'{r.stm_feature} ~ {r.psych_scale}: r={r.r:.2f}, p={r.p_value:.3f}'
            for _, r in sig_rows.iterrows()
        ])

H_StM_ctrl = cond_summary.loc[cond_summary['study_condition']=='control','H_StM_condition'].values[0]
H_StM_def  = cond_summary.loc[cond_summary['study_condition']=='default','H_StM_condition'].values[0]
H_StM_rei  = cond_summary.loc[cond_summary['study_condition']=='reintervention','H_StM_condition'].values[0]

summary_md = f'''# DS5 StM — Model Run Summary

**Run timestamp:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
**Notebook:** notebooks/41_ds5_statistical_mechanics.ipynb
**Model:** StM -- Statistical Mechanics, behavioral Boltzmann analog (Stage 4 Step 1)
**Dataset:** DS5 -- German Adolescents RCT (N=573 total, {len(df)} StoP-eligible)

---

## Critical structural caveat (FLAG A)

The Boltzmann bridge β_behav = 2/σ̂² is an **approximation**. It would be exact for a
potential-well SDE (dX = −(dV/dx)dt + σ dW). The StoP notebook used the constant-drift SDE
(dX = μ dt + σ dW), which has V(x) = −μx (linear, non-confining). The mapping is a behavioral
analog, not a rigorous thermodynamic derivation. Qualitative ordering is robust; quantitative
values are approximate.

## Pipeline parameters

| Parameter | Value |
|---|---|
| MIN_DAYS (inherited from StoP) | {MIN_DAYS} |
| THRESHOLD_USE | {THRESHOLD_USE} |
| KDE_BANDWIDTH | {KDE_BANDWIDTH} |
| KDE_GRID_N | {KDE_GRID_N} |
| SIGMA_THRESHOLD (from DyS) | {SIGMA_THRESHOLD:.6f} |

## FLAG decisions in effect

| Flag | Decision |
|---|---|
| FLAG 1 | Two-state Boltzmann: high-use (X_t > 0.5) vs low-use; single-particle model (J=0) |
| FLAG 2 | β_behav = 2/σ̂² from constant-drift SDE (approximation) |
| FLAG 3 | J = 0: no Ising coupling between DS5 participants |
| FLAG 4 | Phase transition T_c deferred (only 3 discrete conditions) |

## Effective behavioral temperature

| Condition | N | mean β_behav | mean T_behav |
|---|---|---|---|
| control        | {len(df[df.study_condition=='control'])} | {beta_ctrl:.2f} | {1/beta_ctrl:.4f} |
| default        | {len(df[df.study_condition=='default'])} | {beta_def:.2f} | {1/beta_def:.4f} |
| reintervention | {len(df[df.study_condition=='reintervention'])} | {beta_rei:.2f} | {1/beta_rei:.4f} |

β ordering: control > reintervention > default (consistent with σ̂ ordering from StoP).
Friction conditions raise behavioral temperature (more exploration).

## Two-state model & susceptibility

| Condition | mean χ_behav |
|---|---|
| control        | {chi_ctrl:.4f} |
| default        | {chi_def:.4f} |
| reintervention | {chi_rei:.4f} |

## H_StM (condition-level from KDE)

| Condition | H_StM (bits) | H_SDE mean | H_emp mean |
|---|---|---|---|
| control        | {H_StM_ctrl:.3f} | {cond_summary.loc[cond_summary.study_condition=="control","mean_H_SDE"].values[0]:.3f} | {cond_summary.loc[cond_summary.study_condition=="control","mean_H_emp"].values[0]:.3f} |
| default        | {H_StM_def:.3f} | {cond_summary.loc[cond_summary.study_condition=="default","mean_H_SDE"].values[0]:.3f} | {cond_summary.loc[cond_summary.study_condition=="default","mean_H_emp"].values[0]:.3f} |
| reintervention | {H_StM_rei:.3f} | {cond_summary.loc[cond_summary.study_condition=="reintervention","mean_H_SDE"].values[0]:.3f} | {cond_summary.loc[cond_summary.study_condition=="reintervention","mean_H_emp"].values[0]:.3f} |

## Four-model convergence

- N four-model convergent: {n_conv} ({pct_conv:.1f}% of eligible)
- These participants have low σ̂ (StoP), is_dys_eligible (DyS), low H_emp (InT), high β (StM).
- = Thermodynamically ordered, behaviorally entrenched profile.

## Psychosocial correlates (exploratory, friction-exposed, p < 0.05)

  {psych_note}

Total StM psychosocial tests: {len(df_corr)} (no multiple-comparison correction; exploratory)

## Output files

| File | Description |
|---|---|
| participant_stm_parameters.csv | Per-participant StM quantities ({len(df)} rows) |
| condition_stm_summary.csv | Condition-level StM statistics (3 rows) |
| condition_potential.csv | V(x) grid data per condition ({len(df_pot)} rows) |
| stm_stop_int_dys_comparison.csv | Four-model cross-model table ({len(df)} rows) |
| psychosocial_stm_correlations.csv | Pearson r of StM params vs psychosocial scales |
| figures/fig1_stm_beta_by_condition.png | β_behav and T_behav by condition |
| figures/fig2_stm_puse_chi_by_condition.png | Two-state model: p_use and χ by condition |
| figures/fig3_stm_potential_landscape.png | Condition-level V(x) energy landscape |
| figures/fig4_stm_four_model_comparison.png | Four-model comparison scatter |

## Proxy warning

All parameters describe the behavioral proxy X_t = n_opened_day / n_events_day.
X_t approximates behavioral override tendency. It does NOT measure urge, self-control,
or any latent state. All StM analogs (β, T, V, χ, F) inherit this proxy limitation.

## DS5 StM weakness flags (all apply)

- FLAG A: β_behav = 2/σ̂² is an approximation — constant-drift SDE ≠ potential-well SDE
- FLAG B: Individual V(x) estimation underpowered (7-45 observations); condition-level only
- FLAG C: Thermodynamic equilibrium not established (7-45 day windows; possible transients)
- FLAG D: J = 0 only; no Ising coupling between participants
- FLAG E: No phase transitions or T_c estimation (only 3 discrete conditions)
- FLAG F: No hysteresis (no post-intervention behavioral baseline in DS5)
- FLAG G: Behavioral 'energy' is information-theoretic (surprisal), not physical energy

## Forward connections

- StM -> DS4: OU process has confining potential V(x)=(θ/2)(x-μ)²; β = 2θ/σ² is rigorous
- StM -> Phenotyping (100): β_behav, χ_behav, F_behav as three phenotype dimensions
- StM -> Paper §4.1.1: Add Boltzmann bridge, β_behav, Z_2, χ_behav, H_StM, F_behav equations
- StM -> Unification §4.3: StM closes StoP↔DyS↔InT triangle into four-model quadrangle
'''

with open(OUT_DIR / 'model_run_summary.md', 'w') as f:
    f.write(summary_md)
print('Saved model_run_summary.md')


## Interpretation and limits

### Main findings

1. **β_behav is a re-expression of raw σ̂, not evidence that friction raises a physical or psychological temperature:** because raw σ̂ tracks the binomial sampling floor of the daily proportion, the condition ordering is primarily an observation-model diagnostic.

2. **The potential minima are boundary-pinned, not condition-differentiated:** the reflection-corrected KDE places all three minima at the upper boundary because many participant-days have X_t exactly 1. The earlier separation of the minima was boundary leakage, so only distributional spread—not minimum location—is descriptively compared.

3. **The apparent four-model convergence is algebraically dependent:** β is a monotone transform of σ̂, and the DyS low-noise label is thresholded from the same StoP parameters. The overlap is retained as a consistency check, not as four independent confirmations or a stable psychological type.

4. **Susceptibility χ_behav is an exploratory analog:** p_use now follows the declared smoothed P(X_t > 0.5) definition, but β=2/σ̂² inherits the daily-proportion sampling floor. χ therefore cannot be interpreted as a measured psychological susceptibility or an independent empirical dimension.

### Mandatory disclosures (DS5 StM weakness flags)

**FLAG A:** β_behav = 2/σ̂² is an approximation. It would be exact only for a potential-well SDE (dX = −(dV/dx)dt + σ dW). The StoP notebook used the constant-drift SDE (V(x) = −μx, linear, non-confining). The Boltzmann bridge is a behavioral analog, not a rigorous thermodynamic derivation.

**FLAG B:** Individual V(x) estimation from 7–45 daily X_t observations is unreliable. All V(x) analysis uses condition-level pooled data only.

**FLAG C:** Thermodynamic equilibrium is not established. 7–45 day windows may not be long enough for behavioral state distributions to converge to a stationary distribution. Some participants may be in transient adaptation states.

**FLAG D:** No Ising coupling between participants (J = 0). DS5 participants are independent individuals. The Ising model is cited as motivational context only.

**FLAG E:** No phase transitions or critical temperature T_c. Three discrete conditions cannot trace a phase boundary.

**FLAG F:** No hysteresis. DS5 has no post-intervention behavioral baseline. The paper's discussion of hysteresis is motivational only.

**FLAG G:** Behavioral 'energy' is information-theoretic (surprisal = −ln p), not physical energy. β_behav is a dimensionless quantity encoding distribution concentration, not a physical temperature.


## Forward connections and unification

### StoP ↔ DyS ↔ InT ↔ StM — the four-model quadrangle

```
StoP (43): σ̂ (diffusion), μ̂ (drift), x_mean
  │
  ├── σ → 0 ─────────────────────────────── DyS (42): is_dys_eligible, x_mean
  │                                                    fixed-point geometry
  │
  ├── H_SDE = ½ log₂(2πeσ̂²) ──────────── InT (45): H_emp, H_SDE, H_out
  │                                                    three entropy dimensions
  │
  └── β_behav = 2/σ̂²  ────────────────── StM (41): β, T, V(x), χ, F_behav
       [⚠ approximation, FLAG A]                       H_StM, four-model convergence
```

| Bridge | Mechanism | Strength |
|---|---|---|
| StoP → DyS | σ→0 limit: SDE reduces to discrete map | High (exact limit) |
| StoP → InT | H_SDE = ½ log₂(2πeσ̂²): r(H_SDE, H_emp) = 0.650 | High (validated) |
| StoP → StM | β_behav = 2/σ̂² (⚠ constant-drift approximation) | Medium (approximate) |
| InT ↔ StM  | H_StM from Boltzmann KDE triangulates with H_SDE and H_emp | Medium |
| DyS ↔ StM  | DyS-eligible ↔ high β (both from low σ̂; expected r ≈ 1) | High (derived) |

### What StM adds beyond StoP, DyS, InT

1. **Energy cost framing** — Boltzmann distribution: most probable state = lowest-energy state; β controls how steeply energy differences matter
2. **Susceptibility χ** — intervention sensitivity analog; peaks near p_use ≈ 0.5; unique output not available from the other three models
3. **Free energy F_behav** — explicit E−S tradeoff; behavioral locked-in vs exploratory regimes
4. **H_StM** — third entropy estimate for cross-method triangulation (H_SDE / H_StM / H_emp)
5. **Four-model convergence** — closes the triangle into a quadrangle; strongest unification claim in Stage 4 Step 1

### Recommended next steps

- **Paper:** Add 7 missing equations to §4.1.1 / §4.2.1 (see equation gaps cell above)
- **DS4:** OU process gives exact β = 2θ/σ² and rigorous V(x) = (θ/2)(x−μ)²; first dataset where Boltzmann bridge becomes rigorous
- **Step 2 (notebook 100):** β_behav, χ_behav, F_behav as three continuous phenotype dimensions in behavioral profiling
- **StM → SyD:** Energy landscape framing parallels feedback-loop framing in SyD; high-V states maintained by strong reinforcing loops
